# 01 — Data Exploration and Baselines

Multi-label image classification · 12 classes · 128×128 RGB

In [ ]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../experiments")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from collections import Counter

from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset, split_dataset,
    get_train_transform, get_eval_transform, build_dataloaders,
    plot_per_class_examples, plot_multilabel_examples,
    compute_label_prevalence,
    make_topk_predictor, make_random_predictor,
    evaluate_predictor, print_metric_table,
    NUM_LABELS, METRIC_KEYS,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")


In [ ]:
BASE_DIR   = "../data/aggregated"
IMAGE_SIZE = 128
BATCH_SIZE = 128
SPLIT      = [0.7, 0.15, 0.15]


## 1. Data Visualization

In [ ]:
full_dataset = load_dataset(BASE_DIR)
print(f"Total images: {len(full_dataset)}")


In [ ]:
plot_per_class_examples(full_dataset, LABEL_ORDER, per_class=3)

In [ ]:
plot_multilabel_examples(full_dataset, LABEL_ORDER, max_items=9)

## 2. Dataset Statistics

In [ ]:
all_targets = torch.stack([target for _, target in full_dataset])
label_freq  = all_targets.mean(dim=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(LABEL_ORDER, label_freq.numpy())
ax.set_xlabel("Label")
ax.set_ylabel("Frequency")
ax.set_title("Label prevalence")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

df_freq = pd.DataFrame({"label": LABEL_ORDER, "frequency": label_freq.numpy().round(3)})
print(df_freq.sort_values("frequency", ascending=False).to_string(index=False))


In [ ]:
label_counts = Counter(int(t.sum()) for _, t in full_dataset)

fig, ax = plt.subplots(figsize=(8, 4))
xs = sorted(label_counts)
ax.bar(xs, [label_counts[x] for x in xs])
ax.set_xlabel("Labels per image")
ax.set_ylabel("Images")
ax.set_title("Distribution of labels per image")
for x in xs:
    ax.text(x, label_counts[x] + 5, str(label_counts[x]), ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
cooc = (all_targets.T @ all_targets) / len(all_targets)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cooc.numpy(), cmap="Blues", vmin=0, vmax=cooc.max().item())
ax.set_xticks(range(NUM_LABELS))
ax.set_yticks(range(NUM_LABELS))
ax.set_xticklabels(LABEL_ORDER, rotation=45, ha="right")
ax.set_yticklabels(LABEL_ORDER)
ax.set_title("Label co-occurrence (normalized)")
plt.colorbar(im)
plt.tight_layout()
plt.show()


## 3. Data Splitting (70 / 15 / 15)

In [ ]:
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)

train_transform = get_train_transform(IMAGE_SIZE)
eval_transform  = get_eval_transform(IMAGE_SIZE)
train_loader, val_loader, test_loader = build_dataloaders(
    train_raw, val_raw, test_raw, train_transform, eval_transform,
    batch_size=BATCH_SIZE,
)
print(f"Train: {len(train_raw)}  |  Val: {len(val_raw)}  |  Test: {len(test_raw)}")


## 4. Baselines

In [ ]:
def make_prior_weighted_predictor(label_prevalence, num_labels):
    """Each label sampled independently with p = training prevalence."""
    def predict_fn(images, threshold=0.5):
        bsz   = images.shape[0]
        probs = label_prevalence.unsqueeze(0).expand(bsz, -1).to(images.device).clone()
        preds = torch.bernoulli(probs)
        logits = torch.logit(probs.clamp(1e-6, 1 - 1e-6))
        return preds, probs, logits
    return predict_fn

def make_all_positive_predictor(num_labels):
    """Always predicts every class as positive."""
    def predict_fn(images, threshold=0.5):
        bsz   = images.shape[0]
        preds  = torch.ones(bsz, num_labels, device=images.device)
        probs  = torch.full((bsz, num_labels), 0.95, device=images.device)
        logits = torch.logit(probs)
        return preds, probs, logits
    return predict_fn


In [ ]:
prevalence = compute_label_prevalence(train_loader)

baselines = {
    "top-1 freq":      make_topk_predictor(prevalence, 1, NUM_LABELS, DEVICE),
    "top-2 freq":      make_topk_predictor(prevalence, 2, NUM_LABELS, DEVICE),
    "top-3 freq":      make_topk_predictor(prevalence, 3, NUM_LABELS, DEVICE),
    "random uniform":  make_random_predictor(NUM_LABELS),
    "random weighted": make_prior_weighted_predictor(prevalence, NUM_LABELS),
    "all positive":    make_all_positive_predictor(NUM_LABELS),
}

results = {}
for name, fn in baselines.items():
    results[name] = evaluate_predictor(val_loader, fn, DEVICE)
    print_metric_table(f"{name} (val)", results[name])


## 5. Results Summary

In [ ]:
import pandas as pd

rows = []
for name, m in results.items():
    rows.append({"baseline": name, **{k: round(m[k], 4) for k in METRIC_KEYS}})

df = pd.DataFrame(rows).set_index("baseline")
print(df.to_string())
